In [1]:
import pandas as pd
import numpy as np
import time

# ============================================================
# 1. LOAD RAW 5-MIN DATA
# ============================================================

raw = pd.read_csv(
    "ES_full_5min.txt",
    header=None,
    names=["timestamp","open","high","low","close","volume"]
)
raw["timestamp"] = pd.to_datetime(raw["timestamp"])
raw = raw.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded {len(raw)} bars")

# Build timestamp → index lookup
raw_index = {t: i for i, t in enumerate(raw["timestamp"])}


# ============================================================
# 2. SWING HIGHS / LOWS
# ============================================================

SWING_LEN = 3
raw["swing_high"] = False
raw["swing_low"] = False

for i in range(SWING_LEN, len(raw) - SWING_LEN):
    window = raw.iloc[i-SWING_LEN:i+SWING_LEN+1]
    if raw.loc[i, "high"] == window["high"].max():
        raw.at[i, "swing_high"] = True
    if raw.loc[i, "low"] == window["low"].min():
        raw.at[i, "swing_low"] = True

swing_highs = raw[raw["swing_high"]]
swing_lows = raw[raw["swing_low"]]

print(f"Swing highs: {len(swing_highs)}, swing lows: {len(swing_lows)}")


# ============================================================
# 3. BOS DETECTION
# ============================================================

bos_list = []

# Bullish BOS
for i in range(1, len(swing_highs)):
    prev_idx = swing_highs.index[i-1]
    curr_idx = swing_highs.index[i]
    prev_high = raw.loc[prev_idx, "high"]

    segment = raw.loc[prev_idx:curr_idx]
    bos_candidates = segment[segment["close"] > prev_high]
    if not bos_candidates.empty:
        bos_idx = bos_candidates.index[0]
        bos_list.append({
            "bos_idx": bos_idx,
            "direction": "long",
            "broken_level": prev_high,
            "timestamp_bos": raw.loc[bos_idx, "timestamp"]
        })

# Bearish BOS
for i in range(1, len(swing_lows)):
    prev_idx = swing_lows.index[i-1]
    curr_idx = swing_lows.index[i]
    prev_low = raw.loc[prev_idx, "low"]

    segment = raw.loc[prev_idx:curr_idx]
    bos_candidates = segment[segment["close"] < prev_low]
    if not bos_candidates.empty:
        bos_idx = bos_candidates.index[0]
        bos_list.append({
            "bos_idx": bos_idx,
            "direction": "short",
            "broken_level": prev_low,
            "timestamp_bos": raw.loc[bos_idx, "timestamp"]
        })

bos_df = pd.DataFrame(bos_list).sort_values("bos_idx").reset_index(drop=True)
print(f"BOS events: {len(bos_df)}")


# ============================================================
# 4. BREAKER BLOCK IDENTIFICATION
# ============================================================

breaker_setups = []

for _, row in bos_df.iterrows():
    bos_idx = int(row["bos_idx"])
    direction = row["direction"]

    if direction == "long":
        prior_swings = swing_lows[swing_lows.index < bos_idx]
        if prior_swings.empty:
            continue
        brk_idx = prior_swings.index[-1]
    else:
        prior_swings = swing_highs[swing_highs.index < bos_idx]
        if prior_swings.empty:
            continue
        brk_idx = prior_swings.index[-1]

    brk_low = raw.loc[brk_idx, "low"]
    brk_high = raw.loc[brk_idx, "high"]
    if brk_high <= brk_low:
        continue

    breaker_setups.append({
        "bos_idx": bos_idx,
        "direction": direction,
        "breaker_idx": brk_idx,
        "breaker_low": brk_low,
        "breaker_high": brk_high,
        "timestamp_bos": row["timestamp_bos"],
        "timestamp_breaker": raw.loc[brk_idx, "timestamp"]
    })

breaker_df = pd.DataFrame(breaker_setups)
print(f"Breaker setups: {len(breaker_df)}")


# ============================================================
# 5. RETEST + 1R SIMULATION
# ============================================================

RESULT_LOOKAHEAD = 40
RETEST_LOOKAHEAD = 40

results = []

for _, s in breaker_df.iterrows():
    bos_idx = int(s["bos_idx"])
    direction = s["direction"]
    brk_low = s["breaker_low"]
    brk_high = s["breaker_high"]

    if direction == "long":
        entry = brk_high
        stop = brk_low
        R = entry - stop
    else:
        entry = brk_low
        stop = brk_high
        R = stop - entry

    if R <= 0:
        continue

    # Find retest
    retest_idx = None
    for j in range(bos_idx+1, min(bos_idx+RETEST_LOOKAHEAD, len(raw))):
        high = raw.loc[j, "high"]
        low = raw.loc[j, "low"]

        if direction == "long":
            if low <= brk_high and low >= brk_low:
                retest_idx = j
                break
        else:
            if high >= brk_low and high <= brk_high:
                retest_idx = j
                break

    if retest_idx is None:
        continue

    # Simulate 1R
    hit_target = False
    hit_stop = False

    for k in range(retest_idx+1, min(retest_idx+RESULT_LOOKAHEAD, len(raw))):
        high = raw.loc[k, "high"]
        low = raw.loc[k, "low"]

        if direction == "long":
            if low <= stop:
                hit_stop = True
                break
            if high >= entry + R:
                hit_target = True
                break
        else:
            if high >= stop:
                hit_stop = True
                break
            if low <= entry - R:
                hit_target = True
                break

    if hit_target and not hit_stop:
        trade_R = 1
        outcome = "win"
    else:
        trade_R = -1
        outcome = "loss"

    results.append({
        "timestamp_bos": s["timestamp_bos"],
        "timestamp_breaker": s["timestamp_breaker"],
        "timestamp_entry": raw.loc[retest_idx, "timestamp"],
        "direction": direction,
        "breaker_low": brk_low,
        "breaker_high": brk_high,
        "entry": entry,
        "stop": stop,
        "R": trade_R
    })

res = pd.DataFrame(results)
res = res.sort_values("timestamp_entry").reset_index(drop=True)


# ============================================================
# 6. BOS DISPLACEMENT METRICS
# ============================================================

# Attach BOS OHLC
bos_open = []
bos_high = []
bos_low = []
bos_close = []

for t in res["timestamp_bos"]:
    i = raw_index[t]
    bos_open.append(raw.loc[i, "open"])
    bos_high.append(raw.loc[i, "high"])
    bos_low.append(raw.loc[i, "low"])
    bos_close.append(raw.loc[i, "close"])

res["bos_open"] = bos_open
res["bos_high"] = bos_high
res["bos_low"] = bos_low
res["bos_close"] = bos_close

# Displacement metrics
res["bos_range"] = res["bos_high"] - res["bos_low"]
res["bos_body"] = abs(res["bos_close"] - res["bos_open"])
res["bos_body_ratio"] = res["bos_body"] / res["bos_range"]

raw["atr"] = raw["close"].rolling(14).std()
res["atr"] = raw["atr"].iloc[-len(res):].values
res["bos_rel"] = res["bos_range"] / res["atr"]

res["close_beyond"] = np.where(
    res["direction"] == "long",
    res["bos_close"] - res["breaker_low"],
    res["breaker_high"] - res["bos_close"]
)


# ============================================================
# 7. RETEST QUALITY METRICS
# ============================================================

retest_depth = []
wick_only = []
body_inside = []
midpoint_tag = []
full_sweep = []
retest_speed = []
retest_distance_points = []

for _, row in res.iterrows():
    bos_idx = raw_index[row["timestamp_bos"]]
    entry_idx = raw_index[row["timestamp_entry"]]

    # Speed
    retest_speed.append(entry_idx - bos_idx)

    # Breaker zone
    low = row["breaker_low"]
    high = row["breaker_high"]
    mid = (low + high) / 2
    height = high - low

    # Retest candle OHLC
    o = raw.loc[entry_idx, "open"]
    h = raw.loc[entry_idx, "high"]
    l = raw.loc[entry_idx, "low"]
    c = raw.loc[entry_idx, "close"]

    # Depth
    if row["direction"] == "long":
        depth = (high - l) / height
    else:
        depth = (h - low) / height
    retest_depth.append(depth)

    # Wick-only
    if row["direction"] == "long":
        wick_only.append((l <= high) and (l >= low) and (o > high) and (c > high))
    else:
        wick_only.append((h >= low) and (h <= high) and (o < low) and (c < low))

    # Body inside
    if row["direction"] == "long":
        body_inside.append((o <= high and o >= low) or (c <= high and c >= low))
    else:
        body_inside.append((o >= low and o <= high) or (c >= low and c <= high))

    # Midpoint tag
    midpoint_tag.append((l <= mid <= h))

    # Full sweep
    full_sweep.append((l <= low) and (h >= high))

    # Distance in points
    if row["direction"] == "long":
        retest_distance_points.append(row["entry"] - l)
    else:
        retest_distance_points.append(h - row["entry"])

res["retest_depth"] = retest_depth
res["wick_only"] = wick_only
res["body_inside"] = body_inside
res["midpoint_tag"] = midpoint_tag
res["full_sweep"] = full_sweep
res["retest_speed"] = retest_speed
res["retest_distance_points"] = retest_distance_points


# ============================================================
# 8. QUALITY SCORE
# ============================================================

def compute_quality_score(row):
    score = 0

    # Retest quality
    if row["wick_only"]:
        score += 2
    if row["retest_depth"] < 0.25:
        score += 1
    elif row["retest_depth"] < 0.50:
        score += 0.5
    if row["body_inside"]:
        score -= 1
    if row["full_sweep"]:
        score -= 2

    # Displacement
    if row["bos_body_ratio"] > 0.7:
        score += 1
    elif row["bos_body_ratio"] > 0.5:
        score += 0.5
    if row["close_beyond"] > 0.25:
        score += 0.5

    return score

res["quality_score"] = res.apply(compute_quality_score, axis=1)


# ============================================================
# 9. SAVE ENRICHED DATASET
# ============================================================

res.to_csv("ES_5m_Breaker_Enriched.csv", index=False)
print("Saved enriched dataset.")


Loaded 1276757 bars
Swing highs: 208413, swing lows: 203899
BOS events: 91075
Breaker setups: 91021
Saved enriched dataset.


In [2]:
# ============================================================
# 10. LOAD 1-MINUTE DATA
# ============================================================

df1 = pd.read_csv(
    "ES_full_1min.txt",   # <-- use .txt here
    header=None,
    names=["timestamp","open","high","low","close","volume"]
)
df1["timestamp"] = pd.to_datetime(df1["timestamp"])
df1 = df1.sort_values("timestamp").reset_index(drop=True)


print(f"Loaded {len(df1)} 1-minute bars")

# Build lookup
idx_1m = {t: i for i, t in enumerate(df1["timestamp"])}

# ============================================================
# 11. MAP 5-MIN RETEST CANDLE TO ITS 1-MIN CHILDREN
# ============================================================

# Load enriched 5m dataset
res = pd.read_csv("ES_5m_Breaker_Enriched.csv")
res["timestamp_entry"] = pd.to_datetime(res["timestamp_entry"])

# Storage for 1m metrics
one_min_depth = []
one_min_closes_inside = []
one_min_rejection = []
one_min_sweep = []
one_min_vol = []

for _, row in res.iterrows():
    t = row["timestamp_entry"]

    # 5m bar start time
    t0 = t
    # 1m children timestamps
    children = [t0 + pd.Timedelta(minutes=i) for i in range(5)]

    # Extract 1m bars
    bars = []
    for ts in children:
        if ts in idx_1m:
            bars.append(df1.loc[idx_1m[ts]])
    if len(bars) == 0:
        # no data found
        one_min_depth.append(np.nan)
        one_min_closes_inside.append(np.nan)
        one_min_rejection.append(False)
        one_min_sweep.append(False)
        one_min_vol.append(np.nan)
        continue

    bars = pd.DataFrame(bars)

    low = row["breaker_low"]
    high = row["breaker_high"]
    mid = (low + high) / 2
    height = high - low

    # 1m depth
    if row["direction"] == "long":
        depth = (high - bars["low"].min()) / height
    else:
        depth = (bars["high"].max() - low) / height
    one_min_depth.append(depth)

    # 1m closes inside zone
    closes_inside = ((bars["close"] >= low) & (bars["close"] <= high)).sum()
    one_min_closes_inside.append(closes_inside)

    # 1m rejection candle (close back outside zone)
    if row["direction"] == "long":
        rejection = (bars["close"] > high).any()
    else:
        rejection = (bars["close"] < low).any()
    one_min_rejection.append(rejection)

    # 1m sweep (wick through both edges)
    sweep = (bars["low"].min() < low) and (bars["high"].max() > high)
    one_min_sweep.append(sweep)

    # 1m volatility (std of 1m range)
    one_min_vol.append((bars["high"] - bars["low"]).std())

# Attach to dataset
res["one_min_depth"] = one_min_depth
res["one_min_closes_inside"] = one_min_closes_inside
res["one_min_rejection"] = one_min_rejection
res["one_min_sweep"] = one_min_sweep
res["one_min_vol"] = one_min_vol

# Save updated dataset
res.to_csv("ES_5m_Breaker_Enriched_1m.csv", index=False)
print("Saved enriched dataset with 1-minute metrics.")


Loaded 6347253 1-minute bars
Saved enriched dataset with 1-minute metrics.


In [3]:
# ============================================================
# 12. EXECUTION SCORE (1-MINUTE MICROSTRUCTURE)
# ============================================================

import numpy as np
import pandas as pd

# Load enriched dataset with 1m metrics
res = pd.read_csv("ES_5m_Breaker_Enriched_1m.csv")

# Compute global volatility thresholds
vol_30 = res["one_min_vol"].quantile(0.30)
vol_70 = res["one_min_vol"].quantile(0.70)

def compute_execution_score(row):
    score = 0.0

    # --------------------------------------------------------
    # 1. DEPTH (1m precision)
    # --------------------------------------------------------
    if row["one_min_depth"] < 0.25:
        score += 1
    elif row["one_min_depth"] < 0.50:
        score += 0.5
    else:
        score -= 1

    # --------------------------------------------------------
    # 2. CLOSES INSIDE BREAKER ZONE
    # --------------------------------------------------------
    if row["one_min_closes_inside"] == 0:
        score += 1
    elif row["one_min_closes_inside"] == 1:
        score += 0.5
    else:
        score -= 1

    # --------------------------------------------------------
    # 3. 1-MINUTE REJECTION CANDLE
    # --------------------------------------------------------
    if row["one_min_rejection"]:
        score += 1

    # --------------------------------------------------------
    # 4. SWEEP BEHAVIOR
    # --------------------------------------------------------
    if row["one_min_sweep"]:
        score -= 1

    # --------------------------------------------------------
    # 5. VOLATILITY AT RETEST
    # --------------------------------------------------------
    if row["one_min_vol"] < vol_30:
        score += 0.5
    elif row["one_min_vol"] > vol_70:
        score -= 0.5

    return score

# Apply execution score
res["execution_score"] = res.apply(compute_execution_score, axis=1)

# Save updated dataset
res.to_csv("ES_5m_Breaker_Enriched_1m_Scored.csv", index=False)

print("Execution score added and dataset saved.")

Execution score added and dataset saved.


In [4]:
import pandas as pd

res = pd.read_csv("ES_5m_Breaker_Enriched_1m_Scored.csv")

# Combined score = structural + execution
res["combined_score"] = res["quality_score"] + res["execution_score"]

# Bucket the combined score for analysis
res["combined_bucket"] = pd.cut(
    res["combined_score"],
    bins=[-10, 0, 2, 4, 6, 8, 20],
    labels=["<0", "0–2", "2–4", "4–6", "6–8", "8+"]
)

summary = res.groupby("combined_bucket")["R"].agg(
    trades="count",
    win_rate=lambda x: (x > 0).mean(),
    expectancy="mean"
)

print(summary)

                 trades  win_rate  expectancy
combined_bucket                              
<0                12017  0.299659   -0.400682
0–2               10456  0.450842   -0.098317
2–4               11550  0.539134    0.078268
4–6                8716  0.666820    0.333639
6–8               22949  0.711883    0.423766
8+                    0       NaN         NaN


E:\TEMP\ipykernel_31100\2249535549.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = res.groupby("combined_bucket")["R"].agg(


In [5]:
import pandas as pd

# ============================================================
# 13. COMBINED SCORE (STRUCTURE + EXECUTION)
# ============================================================

# Load dataset with execution_score
res = pd.read_csv("ES_5m_Breaker_Enriched_1m_Scored.csv")

# Combined score = structural + execution
res["combined_score"] = res["quality_score"] + res["execution_score"]

# Create score buckets
res["combined_bucket"] = pd.cut(
    res["combined_score"],
    bins=[-10, 0, 2, 4, 6, 8, 20],
    labels=["<0", "0–2", "2–4", "4–6", "6–8", "8+"]
)

# Save updated dataset
res.to_csv("ES_5m_Breaker_Enriched_1m_Scored.csv", index=False)
print("Combined score + bucket added and dataset saved.")

# ============================================================
# 14. TRADES PER YEAR BY SCORE BUCKET
# ============================================================

res["timestamp_entry"] = pd.to_datetime(res["timestamp_entry"])
res["year"] = res["timestamp_entry"].dt.year

trades_per_year = (
    res.groupby(["year", "combined_bucket"])["R"]
    .count()
    .unstack(fill_value=0)
)

print(trades_per_year)

Combined score + bucket added and dataset saved.
combined_bucket   <0  0–2  2–4  4–6   6–8  8+
year                                         
2008             732  572  629  493  1296   0
2009             731  590  668  456  1339   0
2010             634  551  671  439  1416   0
2011             586  555  650  472  1433   0
2012             675  545  675  437  1398   0
2013             631  589  648  493  1388   0
2014             610  570  658  472  1372   0
2015             595  588  646  489  1295   0
2016             641  534  687  473  1334   0
2017             627  565  682  487  1353   0
2018             661  619  634  440  1229   0
2019             622  568  641  485  1285   0
2020             683  605  595  494  1083   0
2021             680  615  626  443  1153   0
2022             736  575  576  533  1124   0
2023             687  606  648  518  1163   0
2024             714  596  601  491  1127   0
2025             710  556  560  551  1061   0
2026              62   57   55 

E:\TEMP\ipykernel_31100\2004704093.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  res.groupby(["year", "combined_bucket"])["R"]


In [6]:
import numpy as np

entry_prices = []
entry_times = []
stop_prices = []
targets = []
outcomes = []   # +1 or -1

for _, row in res.iterrows():
    t0 = row["timestamp_entry"]
    direction = row["direction"]
    low_zone = row["breaker_low"]
    high_zone = row["breaker_high"]

    # 1m children timestamps
    children = [t0 + pd.Timedelta(minutes=i) for i in range(5)]

    # extract 1m bars
    bars = []
    for ts in children:
        if ts in idx_1m:
            bars.append(df1.loc[idx_1m[ts]])
    if len(bars) == 0:
        entry_prices.append(np.nan)
        entry_times.append(pd.NaT)
        stop_prices.append(np.nan)
        targets.append(np.nan)
        outcomes.append(np.nan)
        continue

    bars = pd.DataFrame(bars)

    entry_price = np.nan
    entry_time = pd.NaT
    stop = np.nan
    target = np.nan

    # LONG ENTRY LOGIC
    if direction == "long":
        for i, b in bars.iterrows():
            if b["low"] <= high_zone and b["close"] > high_zone:
                entry_price = b["close"]
                entry_time = b["timestamp"]
                stop = low_zone
                target = entry_price + (entry_price - stop)
                break

    # SHORT ENTRY LOGIC
    else:
        for i, b in bars.iterrows():
            if b["high"] >= low_zone and b["close"] < low_zone:
                entry_price = b["close"]
                entry_time = b["timestamp"]
                stop = high_zone
                target = entry_price - (stop - entry_price)
                break

    # If no entry found
    if np.isnan(entry_price):
        entry_prices.append(np.nan)
        entry_times.append(pd.NaT)
        stop_prices.append(np.nan)
        targets.append(np.nan)
        outcomes.append(np.nan)
        continue

    # Simulate forward on 1m for outcome
    # Look ahead 40 minutes (40 bars)
    start_idx = idx_1m.get(entry_time, None)
    if start_idx is None:
        outcomes.append(np.nan)
        entry_prices.append(entry_price)
        entry_times.append(entry_time)
        stop_prices.append(stop)
        targets.append(target)
        continue

    lookahead = df1.iloc[start_idx:start_idx+40]

    win = False
    loss = False

    if direction == "long":
        for _, b in lookahead.iterrows():
            if b["low"] <= stop:
                loss = True
                break
            if b["high"] >= target:
                win = True
                break
    else:
        for _, b in lookahead.iterrows():
            if b["high"] >= stop:
                loss = True
                break
            if b["low"] <= target:
                win = True
                break

    if win:
        outcomes.append(1)
    elif loss:
        outcomes.append(-1)
    else:
        outcomes.append(0)  # neither hit

    entry_prices.append(entry_price)
    entry_times.append(entry_time)
    stop_prices.append(stop)
    targets.append(target)

# Attach to dataset
res["entry_price_1m"] = entry_prices
res["entry_time_1m"] = entry_times
res["stop_1m"] = stop_prices
res["target_1m"] = targets
res["outcome_1m"] = outcomes

In [7]:
# If you want pure 1R binary outcome:
res["R_1m"] = res["outcome_1m"].replace({1: 1.0, -1: -1.0, 0: 0.0})


In [8]:
# Drop trades with no 1m outcome
valid = res.dropna(subset=["R_1m", "combined_bucket"])

summary_1m = valid.groupby("combined_bucket")["R_1m"].agg(
    trades="count",
    win_rate=lambda x: (x > 0).mean(),
    expectancy="mean"
)

print(summary_1m)


                 trades  win_rate  expectancy
combined_bucket                              
<0                 4435  0.220293   -0.481849
0–2                5761  0.276167   -0.233814
2–4                6721  0.350692   -0.058027
4–6                7753  0.425513    0.093125
6–8               22365  0.429466    0.111379
8+                    0       NaN         NaN


E:\TEMP\ipykernel_31100\519244512.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary_1m = valid.groupby("combined_bucket")["R_1m"].agg(


In [11]:
def get_1m_bars_for_retest(row, df1):
    """
    Given a 5m trade row, return the 5 one-minute bars inside the retest candle.
    """
    t0 = pd.to_datetime(row["timestamp_entry"])

    # The 5-minute candle spans t0, t0+1m, t0+2m, t0+3m, t0+4m
    times = [t0 + pd.Timedelta(minutes=i) for i in range(5)]

    # Select the 1m bars that exist in df1
    bars = df1.loc[df1.index.isin(times)].copy()

    # If fewer than 1–2 bars exist, return empty
    if len(bars) == 0:
        return None

    return bars

In [12]:
def entry_sweep_reclaim(bars, direction, low_zone, high_zone):
    # return entry_price, entry_time, stop, target, outcome
    ...

def entry_two_candle(bars, direction, low_zone, high_zone):
    ...

def entry_midline_reclaim(bars, direction, low_zone, high_zone):
    ...

In [14]:
for _, row in res.iterrows():
    bars = get_1m_bars_for_retest(row, df1)
    if bars is None:
        continue

    # Now you can test:
    # e1 = entry_sweep_reclaim(bars, ...)
    # e2 = entry_two_candle(bars, ...)
    # e3 = entry_midline_reclaim(bars, ...)


KeyboardInterrupt: 